# Módulo 3 · Clase 6 (Práctica) — De las RNN a los LLMs en código
### Deep Learning · Laboratorio

**Objetivos.** Cada estudiante habrá:
1. Tokenizado texto y construido un vocabulario.
2. Entrenado una **LSTM desde cero** para clasificar sentimiento.
3. Hecho **fine-tuning de un modelo tipo BERT** con Hugging Face.
4. Usado un **LLM generativo** y explorado el efecto de la temperatura.

**Agenda (≈ 3 h):**
| Bloque | Tema | ~min |
|---|---|---|
| 0 | Setup y datos | 15 |
| 1 | Tokenización y vocabulario | 25 |
| 2 | Clasificador LSTM desde cero | 45 |
| — | *Descanso* | 10 |
| 3 | Fine-tuning de BERT (Hugging Face) | 45 |
| 4 | Un LLM generativo | 25 |
| 5 | Mini-reto | 15 |


In [49]:
# Bloque 0 · Setup
!pip install -q transformers datasets
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
from torch.utils.data import DataLoader
torch.manual_seed(0); np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


### Dataset: `rotten_tomatoes`
Reseñas de cine etiquetadas como positivas (1) o negativas (0). Es pequeño y va rápido. Usamos un subconjunto para que el lab sea ágil.

In [50]:
from datasets import load_dataset
ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")
# Subconjuntos para agilidad
train_data = ds["train"].shuffle(seed=0).select(range(3000))
test_data  = ds["test"].select(range(1000))

In [51]:
for i, example in enumerate(train_data):
    print(example)
    if i == 10:
        break
print("train:", len(train_data), "| test:", len(test_data))

{'text': "this is pure , exciting moviemaking . you won't exactly know what's happening but you'll be blissfully exhausted .", 'label': 1}
{'text': 'ja rule and kurupt should have gotten to rap . it would have benefitted the dialogue .', 'label': 0}
{'text': 'if you ever wondered what it would be like to be smack in the middle of a war zone armed with nothing but a camera , this oscar-nominated documentary takes you there .', 'label': 1}
{'text': "although there are several truly jolting scares , there's also an abundance of hackneyed dialogue and more silly satanic business than you can shake a severed limb at .", 'label': 0}
{'text': "my wife is an actress is an utterly charming french comedy that feels so american in sensibility and style it's virtually its own hollywood remake .", 'label': 1}
{'text': 'like old myths and wonder tales spun afresh .', 'label': 1}
{'text': "we get some truly unique character studies and a cross-section of americana that hollywood couldn't possibly fic

## Bloque 1 · Tokenización y vocabulario

Construyamos un tokenizador simple (por palabras) y un vocabulario a partir del train. Reservamos dos tokens especiales: `<pad>` (relleno) y `<unk>` (palabra desconocida).

In [53]:
import re
from collections import Counter

def tokenizar(texto):
    return re.findall(r"[a-z']+", texto.lower())

# Construir vocabulario con las palabras más frecuentes
contador = Counter()
for ej in train_data:
  contador.update(tokenizar(ej["text"]))

#print(contador)

vocab = {"<pad>":0, "<unk>":1}

# Convierte cada palabra del contador en un indice
for palabra, _ in contador.most_common(8000):
    vocab[palabra] = len(vocab)

print(vocab)

print("tamaño del vocabulario:", len(vocab))
print("ejemplo:", tokenizar(train_data[0]["text"]))

# Tokeniza un texto (40 tokens). Si la palabra no es reconocidad devuelve <unk>
def codificar(texto, max_len=40):
    ids = [vocab.get(t, 1) for t in tokenizar(texto)][:max_len]
    return ids if ids else [1]

{'<pad>': 0, '<unk>': 1, 'the': 2, 'a': 3, 'and': 4, 'of': 5, 'to': 6, 'is': 7, 'in': 8, 'that': 9, 'it': 10, 'as': 11, 'but': 12, 'with': 13, 'this': 14, 'film': 15, 'for': 16, 'an': 17, 'movie': 18, 'its': 19, "it's": 20, 'be': 21, 'you': 22, 'on': 23, 'by': 24, 'not': 25, 'one': 26, 'about': 27, 'are': 28, 'like': 29, 'from': 30, 'at': 31, 'more': 32, 'have': 33, 'has': 34, 'all': 35, 'than': 36, 'so': 37, 'his': 38, 'if': 39, 'or': 40, 'i': 41, 'out': 42, 'too': 43, 'story': 44, 'who': 45, 'just': 46, 'even': 47, 'comedy': 48, 'will': 49, 'good': 50, 'some': 51, 'no': 52, 'most': 53, 'up': 54, 'what': 55, 'into': 56, 'much': 57, 'time': 58, 'well': 59, 'can': 60, 'characters': 61, 'only': 62, 'little': 63, 'their': 64, 'way': 65, 'very': 66, 'funny': 67, 'never': 68, 'director': 69, 'been': 70, 'may': 71, 'enough': 72, 'when': 73, 'make': 74, 'life': 75, 'made': 76, 'your': 77, 'two': 78, "doesn't": 79, 'any': 80, 'there': 81, 'which': 82, 'us': 83, 'many': 84, 'them': 85, 'first':

In [55]:
# Collate: codifica un batch y lo rellena (padding) a la misma longitud
from torch.nn.utils.rnn import pad_sequence

def collate(batch):
    seqs = [torch.tensor(codificar(b["text"])) for b in batch]
    labels = torch.tensor([b["label"] for b in batch])
    lengths = torch.tensor([len(s) for s in seqs])
    padded = pad_sequence(seqs, batch_first=True, padding_value=0)
    return padded, lengths, labels

train_loader = DataLoader(train_data, batch_size=64, shuffle=True, collate_fn=collate)
test_loader  = DataLoader(test_data,  batch_size=128, collate_fn=collate)
xb, lb, yb = next(iter(train_loader))
print("| batch:", xb.shape, "\n| batch 1:", xb[:1].tolist(), "\n| longitudes:", lb[:5].tolist(), "\n| labels:", yb[:5].tolist())

| batch: torch.Size([64, 38]) 
| batch 1: [[35, 12, 2, 53, 6100, 6101, 125, 282, 14, 6102, 12, 1176, 486, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]] 
| longitudes: [13, 32, 14, 10, 6] 
| labels: [1, 1, 0, 1, 0]


## Bloque 2 · Clasificador LSTM desde cero

Arquitectura: **Embedding → LSTM → capa lineal**. Usamos el último estado oculto como resumen de la reseña para clasificar.

In [29]:
class LSTMClassifier(nn.Module):
    """
    hidden es la dimensión del estado oculto. Por tanto, si un batch tiene 64 textos, y cada texto es
    de longitud 40 (tokens), entonces internamente se habran creado 2560 estados ocultos, y cada estado
    oculto es de dimensión 128.
    """
    def __init__(self, vocab_size, embed_dim=100, hidden=128, n_classes=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm  = nn.LSTM(embed_dim, hidden, batch_first=True)
        self.fc    = nn.Linear(hidden, n_classes)
    def forward(self, x, lengths):
        e = self.embed(x) # [64, 8002, 100]
        # empaquetar para que la LSTM ignore el padding
        packed = nn.utils.rnn.pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        return self.fc(h_n[-1])           # último estado oculto -> clases [64, 128]

model = LSTMClassifier(len(vocab)).to(device)
print("parámetros:", sum(p.numel() for p in model.parameters()))

for name, param in model.named_parameters():
    print(name, param.shape, param.requires_grad)

parámetros: 918218
embed.weight torch.Size([8002, 100]) True
lstm.weight_ih_l0 torch.Size([512, 100]) True
lstm.weight_hh_l0 torch.Size([512, 128]) True
lstm.bias_ih_l0 torch.Size([512]) True
lstm.bias_hh_l0 torch.Size([512]) True
fc.weight torch.Size([2, 128]) True
fc.bias torch.Size([2]) True


In [30]:
# Entrenamiento
def evaluar(model, loader):
    model.eval(); correct=total=0
    with torch.no_grad():
        for x, lengths, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x, lengths).argmax(1)
            correct += (pred==y).sum().item(); total += len(y)
    return correct/total

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(5):
    model.train()
    for x, lengths, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = loss_fn(model(x, lengths), y)
        loss.backward(); opt.step()
    print(f"época {epoch+1} | test acc: {evaluar(model, test_loader):.3f}")

época 1 | test acc: 0.560
época 2 | test acc: 0.602
época 3 | test acc: 0.603
época 4 | test acc: 0.628
época 5 | test acc: 0.663


### 🧩 Ejercicio 1
Prueba tu clasificador con frases tuyas. Escribe una función `predecir(texto)` que devuelva "positivo"/"negativo".

In [58]:
def predecir(texto):
  model.eval()
  x = torch.tensor([codificar(texto)]).to(device)
  lenghts = torch.tensor([x.shape[1]]).to(device)
  with torch.no_grad():
    prob = torch.softmax(model(x, lenghts), dim=1)[0]
    # softmax(logits) -> prob
  return ("positivo" if prob.argmax().item()==1 else "negativo", round(prob.max().item(),2),
          round(prob.min().item(), 2))

texto = "i don'n like this movie" # positivo
print(predecir(texto))

('negativo', 0.8, 0.2)


## Bloque 3 · Fine-tuning de BERT (Hugging Face)

BERT es un modelo transformer de tipo **encoder**, que toma toda la secuencia a la vez. Se preentrenan enmascarando ciertas palabras del texto de entrada.

En vez de entrenar desde cero, partimos de un Transformer **preentrenado** y lo adaptamos. Usamos **DistilBERT** (una versión más liviana y rápida de BERT). El propio modelo trae su **tokenizador** (sub-palabras), así que no construimos vocabulario a mano.

Cada modelo aprendió una tabla de embeddings asociadas a un vocabulario específico.

In [35]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(ckpt)

# Tokenización con el tokenizador del modelo
def tok_batch(batch):
    enc = tokenizer([b["text"] for b in batch], padding=True, truncation=True,
                    max_length=64, return_tensors="pt")
    enc["labels"] = torch.tensor([b["label"] for b in batch])
    return enc

bert_train = DataLoader(train_data, batch_size=32, shuffle=True, collate_fn=tok_batch)
bert_test  = DataLoader(test_data,  batch_size=64, collate_fn=tok_batch)

# Ejemplo de tokenización en sub-palabras
texto = "an unforgettable cinematic experience"

enc = tokenizer(
    texto,
    padding=True,
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

print("input_ids:")
print(enc["input_ids"])

print("\nattention_mask:")
print(enc["attention_mask"])

tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])

print("\ntokens:")
print(tokens)

print("\nToken | ID | Mask")
for tok, idx, mask in zip(tokens, enc["input_ids"][0], enc["attention_mask"][0]):
    print(f"{tok:15s} | {idx.item():5d} | {mask.item()}")

input_ids:
tensor([[  101,  2019,  4895, 29278, 18150, 10880, 21014,  3325,   102]])

attention_mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])

tokens:
['[CLS]', 'an', 'un', '##for', '##get', '##table', 'cinematic', 'experience', '[SEP]']

Token | ID | Mask
[CLS]           |   101 | 1
an              |  2019 | 1
un              |  4895 | 1
##for           | 29278 | 1
##get           | 18150 | 1
##table         | 10880 | 1
cinematic       | 21014 | 1
experience      |  3325 | 1
[SEP]           |   102 | 1


In [59]:
# Cargar el modelo con una cabeza de clasificación (2 clases) y hacer fine-tuning
model_bert = AutoModelForSequenceClassification.from_pretrained(ckpt, num_labels=2).to(device)
opt = torch.optim.AdamW(model_bert.parameters(), lr=2e-5)

def evaluar_bert(model, loader):
    model.eval(); correct=total=0
    with torch.no_grad():
        for batch in loader:
            batch = {k:v.to(device) for k,v in batch.items()}
            logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
            correct += (logits.argmax(1)==batch["labels"]).sum().item(); total += len(batch["labels"])
    return correct/total

for epoch in range(2):       # 2 épocas bastan al partir de un modelo preentrenado
    model_bert.train()
    for batch in bert_train:
        batch = {k:v.to(device) for k,v in batch.items()} # [[texto], [texto]]
        opt.zero_grad()
        out = model_bert(**batch)        # al pasar labels, devuelve la loss
        out.loss.backward(); opt.step()
    print(f"época {epoch+1} | test acc: {evaluar_bert(model_bert, bert_test):.3f}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


época 1 | test acc: 0.831
época 2 | test acc: 0.819


Fíjate cómo DistilBERT supera a la LSTM con solo **2 épocas**: el preentrenamiento sobre enormes cantidades de texto ya le dio un fuerte conocimiento del lenguaje.

### 🧩 Ejercicio 2
Escribe `predecir_bert(texto)` usando el tokenizador y el modelo afinado, y pruébala con tus frases.

In [2]:
"""
batch :

[[[1], [2], [3]],
 [1, 0, 1]]

texto = Texto en crudo
"""

def predecir_bert(texto):
  model_bert.eval()
  enc = tokenizer(
      texto,
      truncation=True,
      max_length=64,
      return_tensors="pt"
  ).to(device)
  with torch.no_grad():
    prob = torch.softmax(model_bert(**enc).logits, dim=1)[0]
  return ("positivo" if prob.argmax().item() == 1 else "negativo", round(prob.max().item(),2),
          round(prob.min().item(), 2))

texto = "i don't like this movie" # positivo
print(predecir_bert(texto))

NameError: name 'model_bert' is not defined

## Bloque 4 · Un LLM generativo

Hasta ahora clasificamos (comprensión). Ahora **generamos** texto con un modelo autoregresivo solo-decoder. Usamos **DistilGPT-2** (pequeño) vía el `pipeline` de Hugging Face.

En los modelos decoder, cada token puede prestar atención a sí mismo y a tokens anteriores, pero no a tokens futuros (por que son los que queremos generar).

In [37]:
from transformers import pipeline
generador = pipeline("text-generation", model="distilgpt2",
                     device=0 if device=="cuda" else -1)

prompt = "Deep learning is"
salida = generador(prompt, max_new_tokens=40, do_sample=False)   # greedy
print(salida[0]["generated_text"])

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_cor

Deep learning is a great way to learn about the world and how to learn about the world.


























### El efecto de la temperatura
La **temperatura** controla cuán "creativo" es el muestreo: baja → conservador y repetitivo; alta → diverso y arriesgado. Compáralo.

In [38]:
for temp in [0.3, 0.7, 1.2]:
    out = generador("Once upon a time", max_new_tokens=30, do_sample=True,
                    temperature=temp, top_k=50)
    print(f"\n--- temperatura {temp} ---")
    print(out[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample', 'top_k'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- temperatura 0.3 ---
Once upon a time of war, the United States would have a strong, stable and prosperous empire. The United States would have a strong, stable and prosperous empire. The

--- temperatura 0.7 ---
Once upon a time, we were in the middle of the night and it was the night of the day. We were all looking forward to the night and we were all

--- temperatura 1.2 ---
Once upon a time-line by my own husband I always knew that people were good looking; and all this, it took me two, three or four years for many


### 🧩 Ejercicio 3
Los LLMs generativos resuelven tareas **mediante prompting**, sin reentrenar. Diseña un prompt *few-shot* que induzca al modelo a clasificar sentimiento (dale 2-3 ejemplos en el prompt y luego una frase nueva). ¿Funciona con un modelo tan pequeño? Comenta las limitaciones.

In [48]:
# @title Solución (ejemplo de prompt few-shot)
prompt = (
    "Review: I loved this film. Sentiment: positive\n"
    "Review: Terrible and boring. Sentiment: negative\n"
    "Review: A beautiful, moving story. Sentiment:"
)
print(generador(prompt, max_new_tokens=2, do_sample=False)[0]["generated_text"])
print("\nNota: distilgpt2 es muy pequeño; el few-shot suele fallar.")
print("A esta técnica se le llama in-context learning. Recibe ejemplo y trata de imitar el patrón.")
print("Modelos grandes (GPT-4, Claude, Llama) hacen esto de forma fiable: es el poder de la escala.")

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=2) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Review: I loved this film. Sentiment: positive
Review: Terrible and boring. Sentiment: negative
Review: A beautiful, moving story. Sentiment: positive


Nota: distilgpt2 es muy pequeño; el few-shot suele fallar.
A esta técnica se le llama in-context learning. Recibe ejemplo y trata de imitar el patrón.
Modelos grandes (GPT-4, Claude, Llama) hacen esto de forma fiable: es el poder de la escala.


## Bloque 5 · 🏁 Mini-reto

Elige uno:
- **Mejora la LSTM:** bidireccional (`bidirectional=True`), más capas, dropout, embeddings preentrenados (GloVe). ¿Cuánta accuracy logras?
- **Fine-tuning eficiente:** aplica **LoRA** a DistilBERT con la librería `peft` y compara la accuracy y el número de parámetros entrenables contra el fine-tuning completo.

```python
# Pista para LoRA:
# !pip install peft
# from peft import LoraConfig, get_peft_model
# config = LoraConfig(task_type="SEQ_CLS", r=8, lora_alpha=16, target_modules=["q_lin","v_lin"])
# model_lora = get_peft_model(model_bert_base, config)
# model_lora.print_trainable_parameters()
```

In [ ]:
# TODO: tu solución
...

## Cierre

Recorriste el NLP práctico de punta a punta: tokenización, una LSTM desde cero, fine-tuning de un Transformer preentrenado y generación con un LLM.

**En el Módulo 4** llegamos a los temas avanzados: modelos generativos, multimodales y agentes.